#Chile - Universidad Adolfo Ibáñez (UAI)
##Curso NLP
### Text Summarisation

# Text Summarisation (Pipeline)

In [ ]:
# Instalar Librerías
!pip install -U transformers datasets evaluate --quiet

In [ ]:
#Importar Librerías
from transformers import pipeline

In [ ]:
#Cargar el pipeline de QA (Español)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda:0


In [ ]:
#Texto Original
text = """
The World Health Organization (WHO) released a new report on global health trends in 2023.
According to the report, non-communicable diseases such as diabetes and cardiovascular conditions continue to rise,
particularly in low- and middle-income countries. Meanwhile, mental health has become an urgent priority for global public health,
with significant gaps in access to care and resources. The WHO recommends integrated policy frameworks that address both physical
and mental health, alongside new digital health technologies to enhance accessibility and early diagnosis.
"""

In [ ]:
#Summarisation


In [ ]:
#Revisar Resumen
print("📝 Texto Original:\n", text)
print("\n📄 Resumen:\n", summary[0]["summary_text"])

📝 Texto Original:
 
The World Health Organization (WHO) released a new report on global health trends in 2023.
According to the report, non-communicable diseases such as diabetes and cardiovascular conditions continue to rise,
particularly in low- and middle-income countries. Meanwhile, mental health has become an urgent priority for global public health,
with significant gaps in access to care and resources. The WHO recommends integrated policy frameworks that address both physical
and mental health, alongside new digital health technologies to enhance accessibility and early diagnosis.


📄 Resumen:
 The World Health Organization (WHO) released a new report on global health trends in 2023. Non-communicable diseases such as diabetes and cardiovascular conditions continue to rise. Meanwhile, mental health has become an urgent priority for global public health.


# Fine-Tuning Modelo QA (Inglés)

In [ ]:
#Importar Librerías
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer
import evaluate
import numpy as np
import torch

In [ ]:
#Cargar Dataset
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:1%]").train_test_split(test_size=0.2)

In [ ]:
#Revisar Ejemplo (Texto Original)
print("📰 Texto original:\n", dataset["train"][0]["article"])

📰 Texto original:
 WASHINGTON (CNN) -- For the first time since media coverage was banned in 1991, the return of the body of a fallen member of the U.S. armed forces was opened to news outlets late Sunday. A transport plane carries caskets of U.S. servicemen in this photo the Pentagon released in 2005. The U.S. Air Force informed media on Sunday that the family of Staff Sgt. Phillip Myers consented to allowing coverage of his casket being returned to Dover Air Force Base in Delaware. Myers, 30, of Hopewell, Virginia, was a member of an engineering unit based in Britain. He died Saturday in a roadside bombing in southern Afghanistan, the U.S. military reported. In February, President Obama and Defense Secretary Robert Gates overturned a policy that dated back to the first Persian Gulf war. They agreed to allow reporters to observe the remains of American troops being returned to the U.S. military mortuary at Dover, as long as families agreed. The policy was supposed to take effect on Mo

In [ ]:
#Revisar Ejemplo (Resumen)
print("\n📄 Resumen de referencia:\n", dataset["train"][0]["highlights"])


📄 Resumen de referencia:
 Family of Staff Sgt. Phillip Myers consents to coverage of his casket's return .
Body of Myers brought to Dover Air Force Base in Delaware on Sunday night .
This is first time that media coverage has been allowed since ban in 1991 .
In February, President Obama and Defense Secretary Robert Gates overturned policy .


In [ ]:
#Cargar Tokenizador & Modelo


In [ ]:
#Función Pre-Procesamiento
def preprocess(example):
    model_inputs = tokenizer(example["article"], max_length=512, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(example["highlights"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
#Tokenización & Pre-Procesamiento


Map:   0%|          | 0/2296 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4006: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/575 [00:00<?, ? examples/s]

In [ ]:
#Configurar Entrenamiento
training_args = TrainingArguments(
    output_dir="./summarisation_finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=2,
    logging_steps=100,
    report_to="none"
)

In [ ]:
#Crear Data Collator


In [ ]:
#Instalar & Importar Librería
!pip install rouge_score --quiet
import evaluate
import numpy as np

In [ ]:
#Crear Set Evaluación (Pequeño)
eval_set = tokenized_dataset["test"].select(range(50))
eval_set = eval_set.map(preprocess, batched=True)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4006: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
#Configurar Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=eval_set,
    data_collator=data_collator
)

In [ ]:
#Fine-Tuning Modelo
trainer.train()

Step,Training Loss


In [ ]:
#Preparar Inputs (Nuevo Ejemplo)
sample_text = dataset["test"][0]["article"]
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=512).to(model.device)

In [ ]:
#Generar Resumen (Nuevo Ejemplo)
summary_ids = model.generate(**inputs, max_length=100)
print("📰 Texto original:\n", sample_text)
print("\n📄 Resumen generado:\n", tokenizer.decode(summary_ids[0], skip_special_tokens=True))